In [8]:
import pandas as pd
import numpy as np

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    KBinsDiscretizer,
    OneHotEncoder,
    PowerTransformer,
    StandardScaler,
)
from sklearn.impute import SimpleImputer

from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    cross_validate,
    train_test_split,
)

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB, BernoulliNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    ConfusionMatrixDisplay,
    confusion_matrix,
    f1_score,
    make_scorer,
    precision_score,
    recall_score,
    roc_auc_score,
)

import warnings; warnings.filterwarnings("ignore")

## Read Data CSV

In [9]:
sessions = pd.read_csv("Raw_Data/challenge2_train.csv")

display(sessions.head())
sessions.info()

print()
display(sessions["Revenue"].value_counts(dropna=False))
print()
display(sessions["Revenue"].value_counts(normalize=True, dropna=False))

,ID,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,1,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,False,False
1,3,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,False,False
2,4,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,Feb,3,2,2,4,Returning_Visitor,False,False
3,7,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.4,Feb,2,4,3,3,Returning_Visitor,False,False
4,9,0,0.0,0,0.0,2,37.000000,0.00,0.10,0.0,0.8,Feb,2,2,2,3,Returning_Visitor,False,False


<class 'pandas.DataFrame'>
RangeIndex: 9247 entries, 0 to 9246
Data columns (total 19 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   ID                       9247 non-null   int64  
 1   Administrative           9247 non-null   int64  
 2   Administrative_Duration  9247 non-null   float64
 3   Informational            9247 non-null   int64  
 4   Informational_Duration   9247 non-null   float64
 5   ProductRelated           9247 non-null   int64  
 6   ProductRelated_Duration  9247 non-null   float64
 7   BounceRates              9247 non-null   float64
 8   ExitRates                9247 non-null   float64
 9   PageValues               9247 non-null   float64
 10  SpecialDay               9247 non-null   float64
 11  Month                    9247 non-null   str    
 12  OperatingSystems         9247 non-null   int64  
 13  Browser                  9247 non-null   int64  
 14  Region                   9247 non-n

Revenue
False    7816
True     1431
Name: count, dtype: int64

Revenue
False    0.845247
True     0.154753
Name: proportion, dtype: float64

## Train/Validation Splits 

In [10]:
X = sessions.drop(columns=["Revenue", "ID"])
y = sessions["Revenue"].copy()

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

display(pd.DataFrame({
    "development": y_train.value_counts(normalize=True),
    "holdout": y_valid.value_counts(normalize=True),
}))

,development,holdout
Revenue,,
False,0.845208,0.845405
True,0.154792,0.154595


## Model Pipeline

In [11]:
categorical = [
    "Month",
    "OperatingSystems",
    "Browser",
    "Region",
    "TrafficType",
    "VisitorType",
    "Weekend",
]

numerical = [column for column in X_train.columns if column not in categorical]

# 2. Build leakage-free preprocessing per column type
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    # Yeo-Johnson maps skewed continuous features to Gaussian bell curves
    ('power', PowerTransformer(method='yeo-johnson', standardize=True))
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    # Group rare categories (<1% frequency) to prevent zero-variance overfitting
    ('ohe', OneHotEncoder(
        handle_unknown='infrequent_if_exist',
        min_frequency=0.01,
        sparse_output=False
    ))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_pipeline, numerical),
        ('cat', cat_pipeline, categorical)
    ],
    remainder='drop'
)

### Logistic Regression

In [12]:
# --- 1. Cross-Validation & Scoring Setup ---
# Maintain class proportions (~84.5% False / ~15.5% True) across 5 folds
# and optimize specifically for the positive class (Revenue=True) F1 score
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
f1_purchase = make_scorer(f1_score, pos_label=True, zero_division=0)

# --- 2. Dedicated Logistic Regression Base Pipeline ---
# Clone the global preprocessor so fitting GridSearchCV never mutates shared pipelines
logistic_pipeline = Pipeline([
    ("prep", clone(preprocessor)),
    ("model", LogisticRegression(max_iter=2000, random_state=42)),
])

# --- 3. Multi-Regime Regularization & Class-Weight Grid ---
# Dictionary 1 tests L1 (feature-selecting) and L2 (ridge) penalties across liblinear and saga.
# Dictionary 2 tests ElasticNet (combined L1 + L2) with saga to handle collinear numerical pairs.
# Both grids sweep custom class_weight ratios to balance precision and recall on imbalanced data.
param_grid = [
    {
        "model__solver": ["liblinear", "saga"],
        "model__penalty": ["l1", "l2"],
        "model__C": [0.001, 0.01, 0.05, 0.1, 0.5, 1.0, 5.0, 10.0],
        "model__class_weight": [
            None,
            "balanced",
            {False: 1, True: 2},
            {False: 1, True: 3},
            {False: 1, True: 4},
        ],
    },
    {
        "model__solver": ["saga"],
        "model__penalty": ["elasticnet"],
        "model__l1_ratio": [0.25, 0.5, 0.75],
        "model__C": [0.01, 0.1, 0.5, 1.0, 5.0],
        "model__class_weight": [
            None,
            "balanced",
            {False: 1, True: 2},
            {False: 1, True: 3},
            {False: 1, True: 4},
        ],
    },
]

# --- 4. Configure and Fit GridSearchCV Exclusively on Training Data ---
# Fit strictly on X_train to prevent validation data leakage; refit=True automatically
# retrains the winning hyperparameter combination on the full training set
logistic_search = GridSearchCV(
    estimator=logistic_pipeline,
    param_grid=param_grid,
    scoring=f1_purchase,
    cv=cv,
    refit=True,
    return_train_score=True,
    n_jobs=-1,
)
logistic_search.fit(X_train, y_train)

# --- 5. Cross-Validation Performance Summary ---
# Compare mean CV training vs. validation F1 to check generalization before holdout evaluation
best_index = logistic_search.best_index_
best_cv_train_f1 = logistic_search.cv_results_["mean_train_score"][best_index]
best_cv_valid_f1 = logistic_search.best_score_

print("Best Logistic Regression Parameters:", logistic_search.best_params_)
print(f"Best Mean CV Training F1:           {best_cv_train_f1:.3f}")
print(f"Best Mean CV Validation F1:         {best_cv_valid_f1:.3f}")

# --- 6. Learned Feature Weight Inspection ---
# Extract the refitted winning pipeline and display the top coefficients sorted by magnitude
best_logistic_pipe = logistic_search.best_estimator_
feature_names = best_logistic_pipe.named_steps["prep"].get_feature_names_out()
weights = best_logistic_pipe.named_steps["model"].coef_[0]

logistic_weights = (
    pd.DataFrame({
        "Feature": feature_names,
        "Weight": weights,
    })
    .assign(Abs_Weight=lambda df: df["Weight"].abs())
    .sort_values(by="Abs_Weight", ascending=False)
    .drop(columns=["Abs_Weight"])
    .reset_index(drop=True)
)

display(logistic_weights.head(15).round(3))

Best Logistic Regression Parameters: {'model__C': 0.1, 'model__class_weight': {False: 1, True: 2}, 'model__penalty': 'l1', 'model__solver': 'saga'}
Best Mean CV Training F1:           0.694
Best Mean CV Validation F1:         0.692


,Feature,Weight
0,num__PageValues,1.570
1,cat__Month_Nov,1.011
2,cat__Month_May,-0.465
3,cat__TrafficType_1,-0.460
4,cat__VisitorType_Returning_Visitor,-0.402
5,num__ExitRates,-0.284
6,cat__TrafficType_3,-0.255
7,cat__Month_Mar,-0.254
8,cat__Month_Sep,0.249
9,cat__TrafficType_6,-0.203


In [13]:
# --- 1. Generate Predictions on Training and Holdout Sets ---
# Use the refitted best Logistic Regression pipeline for class and probability predictions
y_train_pred = best_logistic_pipe.predict(X_train)
y_valid_pred = best_logistic_pipe.predict(X_valid)
y_valid_prob = best_logistic_pipe.predict_proba(X_valid)[:, 1]

# --- 2. Build Classification Metrics Comparison Table ---
# Evaluate Accuracy, Precision, Recall, and F1 Score side-by-side on train vs. validation
metrics_comparison = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score"],
    "Training": [
        accuracy_score(y_train, y_train_pred),
        precision_score(y_train, y_train_pred, zero_division=0),
        recall_score(y_train, y_train_pred, zero_division=0),
        f1_score(y_train, y_train_pred, zero_division=0),
    ],
    "Validation": [
        accuracy_score(y_valid, y_valid_pred),
        precision_score(y_valid, y_valid_pred, zero_division=0),
        recall_score(y_valid, y_valid_pred, zero_division=0),
        f1_score(y_valid, y_valid_pred, zero_division=0),
    ],
})

display(metrics_comparison.round(3))

,Metric,Training,Validation
0,Accuracy,0.893,0.872
1,Precision,0.624,0.566
2,Recall,0.780,0.752
3,F1 Score,0.694,0.646


### Naïve Bayes

In [14]:
# --- 1. Cross-Validation & Scoring Setup ---
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
f1_purchase = make_scorer(f1_score, pos_label=True, zero_division=0)

# --- 2. Dedicated Naïve Bayes Pipeline & Discretized Numerical Sub-Pipeline ---
# Clone the global preprocessor so overriding 'prep__num' never mutates shared pipelines
Bayes_pipe = Pipeline([
    ("prep", clone(preprocessor)),
    ("clf", GaussianNB()),
])

discretized_num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("bins", KBinsDiscretizer(encode="onehot-dense", subsample=None)),
])

# --- 3. Multi-Regime Hyperparameter Grid ---
# Regime A: Discretized numerical bins + BernoulliNB (binarize=None)
# Regime B: Default PowerTransformer + BernoulliNB across non-negative z-score cutoffs
# Regime C: Default PowerTransformer + continuous GaussianNB
param_grid = [
    {
        "prep__num": [discretized_num_pipeline],
        "prep__num__bins__n_bins": [3, 5, 8, 10],
        "prep__num__bins__strategy": ["quantile", "uniform"],
        "clf": [BernoulliNB(binarize=None)],
        "clf__alpha": [0.01, 0.1, 0.5, 1.0, 2.0],
        "clf__class_prior": [None, [0.8, 0.2], [0.7, 0.3], [0.6, 0.4], [0.5, 0.5]],
    },
    {
        "clf": [BernoulliNB()],
        "clf__alpha": [0.01, 0.1, 0.5, 1.0, 2.0],
        "clf__binarize": [0.0, 0.25, 0.5, 1.0],
        "clf__class_prior": [None, [0.8, 0.2], [0.7, 0.3], [0.6, 0.4], [0.5, 0.5]],
    },
    {
        "clf": [GaussianNB()],
        "clf__var_smoothing": np.logspace(-11, -1, 11),
        "clf__priors": [None, [0.8, 0.2], [0.7, 0.3], [0.6, 0.4], [0.5, 0.5]],
    },
]

# --- 4. Fit GridSearchCV Exclusively on Training Data ---
bayes_search = GridSearchCV(
    estimator=Bayes_pipe,
    param_grid=param_grid,
    scoring=f1_purchase,
    cv=cv,
    refit=True,
    return_train_score=True,
    n_jobs=-1,
)
bayes_search.fit(X_train, y_train)

# --- 5. Cross-Validation Performance Summary ---
best_index = bayes_search.best_index_
best_cv_train_f1 = bayes_search.cv_results_["mean_train_score"][best_index]
best_cv_valid_f1 = bayes_search.best_score_

print("Best GridSearch Parameters:", bayes_search.best_params_)
print(f"Best Mean CV Training F1:   {best_cv_train_f1:.3f}")
print(f"Best Mean CV Validation F1: {best_cv_valid_f1:.3f}")

# --- 6. Dynamic Learned Parameter Inspection ---
best_bayes_pipe = bayes_search.best_estimator_
feature_names = best_bayes_pipe.named_steps["prep"].get_feature_names_out()
best_clf = best_bayes_pipe.named_steps["clf"]

if isinstance(best_clf, GaussianNB):
    learned_params = pd.DataFrame({
        "Feature": feature_names,
        "Mean (y=0)": best_clf.theta_[0],
        "Mean (y=1)": best_clf.theta_[1],
        "Var (y=0)": best_clf.var_[0],
        "Var (y=1)": best_clf.var_[1],
    })
else:
    feature_prob = np.exp(best_clf.feature_log_prob_)
    learned_params = pd.DataFrame({
        "Feature": feature_names,
        "P(Feature=1 | y=0)": feature_prob[0],
        "P(Feature=1 | y=1)": feature_prob[1],
    })

display(learned_params.head(10).round(3))

Best GridSearch Parameters: {'clf': BernoulliNB(), 'clf__alpha': 1.0, 'clf__binarize': 1.0, 'clf__class_prior': [0.8, 0.2]}
Best Mean CV Training F1:   0.648
Best Mean CV Validation F1: 0.651


,Feature,P(Feature=1 | y=0),P(Feature=1 | y=1)
0,num__Administrative,0.235,0.383
1,num__Administrative_Duration,0.213,0.343
2,num__Informational,0.190,0.325
3,num__Informational_Duration,0.171,0.298
4,num__ProductRelated,0.140,0.275
5,num__ProductRelated_Duration,0.129,0.255
6,num__BounceRates,0.214,0.029
7,num__ExitRates,0.224,0.024
8,num__PageValues,0.106,0.820
9,num__SpecialDay,0.117,0.039


In [15]:
# Predictions for training and validation sets using the tuned best pipeline
y_train_pred = best_bayes_pipe.predict(X_train)
y_valid_pred = best_bayes_pipe.predict(X_valid)
y_valid_prob = best_bayes_pipe.predict_proba(X_valid)[:, 1]

# Compare the main classification metrics
metrics_comparison = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score"],
    "Training": [
        accuracy_score(y_train, y_train_pred),
        precision_score(y_train, y_train_pred, zero_division=0),
        recall_score(y_train, y_train_pred, zero_division=0),
        f1_score(y_train, y_train_pred, zero_division=0),
    ],
    "Validation": [
        accuracy_score(y_valid, y_valid_pred),
        precision_score(y_valid, y_valid_pred, zero_division=0),
        recall_score(y_valid, y_valid_pred, zero_division=0),
        f1_score(y_valid, y_valid_pred, zero_division=0),
    ],
})

display(metrics_comparison.round(3))

,Metric,Training,Validation
0,Accuracy,0.866,0.847
1,Precision,0.547,0.503
2,Recall,0.795,0.776
3,F1 Score,0.648,0.611


### K-Nearest Neighbors

In [16]:
# --- 1. Cross-Validation & Scoring Setup ---
# Maintain class proportions (~84.5% False / ~15.5% True) across 5 folds
# and optimize specifically for the positive class (Revenue=True) F1 score
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
f1_purchase = make_scorer(f1_score, pos_label=True, zero_division=0)

# --- 2. Dedicated K-Nearest Neighbors Base Pipeline ---
# Clone the global preprocessor so fitting GridSearchCV never mutates shared pipelines
knn_pipeline = Pipeline([
    ("prep", clone(preprocessor)),
    ("model", KNeighborsClassifier()),
])

# --- 3. Neighborhood Size, Weighting, & Distance Metric Grid ---
# Sweep odd n_neighbors to prevent binary voting ties and smooth high-variance boundaries.
# Test distance weighting to prevent distant majority neighbors from swamping close positives,
# and compare Manhattan, Euclidean, and Cosine metrics in the 59-dimensional mixed feature space.
param_grid = {
    "model__n_neighbors": [3, 5, 7, 9, 11, 15, 19, 25, 31],
    "model__weights": ["uniform", "distance"],
    "model__metric": ["manhattan", "euclidean", "cosine"],
}

# --- 4. Configure and Fit GridSearchCV Exclusively on Training Data ---
# Fit strictly on X_train to prevent validation data leakage; refit=True automatically
# retrains the winning hyperparameter combination on the full training set
knn_search = GridSearchCV(
    estimator=knn_pipeline,
    param_grid=param_grid,
    scoring=f1_purchase,
    cv=cv,
    refit=True,
    return_train_score=True,
    n_jobs=-1,
)
knn_search.fit(X_train, y_train)

# --- 5. Cross-Validation Performance Summary ---
# Compare mean CV training vs. validation F1 to check generalization before holdout evaluation
best_index = knn_search.best_index_
best_cv_train_f1 = knn_search.cv_results_["mean_train_score"][best_index]
best_cv_valid_f1 = knn_search.best_score_

print("Best K-Nearest Neighbors Parameters:", knn_search.best_params_)
print(f"Best Mean CV Training F1:           {best_cv_train_f1:.3f}")
print(f"Best Mean CV Validation F1:         {best_cv_valid_f1:.3f}")

# --- 6. Nearest-Neighbor Inspection on Sample Session ---
# Extract the refitted winning pipeline and inspect the distances and purchase labels
# of the nearest neighbors for the first training session
best_knn_pipe = knn_search.best_estimator_
knn_model = best_knn_pipe.named_steps["model"]

session = X_train.iloc[[0]]
transformed_session = best_knn_pipe.named_steps["prep"].transform(session)
distances, indices = knn_model.kneighbors(transformed_session)

display(pd.DataFrame({
    "Distance": distances[0],
    "Purchased": y_train.iloc[indices[0]].to_numpy(),
}))

Best K-Nearest Neighbors Parameters: {'model__metric': 'euclidean', 'model__n_neighbors': 31, 'model__weights': 'distance'}
Best Mean CV Training F1:           1.000
Best Mean CV Validation F1:         0.665


,Distance,Purchased
0,0.000000,False
1,1.568383,False
2,1.770997,False
3,1.857021,False
4,1.955126,False
5,2.032179,False
6,2.050457,True
7,2.057456,False
8,2.111908,False
9,2.139702,False


In [17]:
# --- 1. Generate Predictions on Training and Holdout Sets ---
# Use the refitted best K-Nearest Neighbors pipeline for class and probability predictions
y_train_pred = best_knn_pipe.predict(X_train)
y_valid_pred = best_knn_pipe.predict(X_valid)
y_valid_prob = best_knn_pipe.predict_proba(X_valid)[:, 1]

# --- 2. Build Classification Metrics Comparison Table ---
# Evaluate Accuracy, Precision, Recall, and F1 Score side-by-side on train vs. validation
metrics_comparison = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score"],
    "Training": [
        accuracy_score(y_train, y_train_pred),
        precision_score(y_train, y_train_pred, zero_division=0),
        recall_score(y_train, y_train_pred, zero_division=0),
        f1_score(y_train, y_train_pred, zero_division=0),
    ],
    "Validation": [
        accuracy_score(y_valid, y_valid_pred),
        precision_score(y_valid, y_valid_pred, zero_division=0),
        recall_score(y_valid, y_valid_pred, zero_division=0),
        f1_score(y_valid, y_valid_pred, zero_division=0),
    ],
})

display(metrics_comparison.round(3))

,Metric,Training,Validation
0,Accuracy,1.0,0.883
1,Precision,1.0,0.633
2,Recall,1.0,0.584
3,F1 Score,1.0,0.607


## Support Vector Machine

In [23]:
# --- 1. Cross-Validation & Scoring Setup ---
# Maintain class proportions (~84.5% False / ~15.5% True) across 5 folds
# and optimize specifically for the positive class (Revenue=True) F1 score
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
f1_purchase = make_scorer(f1_score, pos_label=True, zero_division=0)

# --- 2. Dedicated Support Vector Machine Base Pipeline ---
# Clone the global preprocessor so fitting GridSearchCV never mutates shared pipelines;
# leave probability=False (default) to avoid 5x Platt-scaling overhead during CV
svm_pipe = Pipeline([
    ("prep", clone(preprocessor)),
    ("model", SVC(random_state=42)),
])

# --- 3. Multi-Kernel Hyperparameter Grid (Linear & RBF) ---
# Dictionary 1 tests linear margins across soft-margin C penalties and class weights.
# Dictionary 2 tests non-linear radial basis margins across C, gamma, and class weights.
# GridSearchCV evaluates both kernels and retains only the single best F1 model.
param_grid = [
    {
        "model__kernel": ["linear"],
        "model__C": [0.01, 0.1, 0.5, 1.0, 2.0, 5.0],
        "model__class_weight": [
            None,
            "balanced",
            {False: 1, True: 2},
            {False: 1, True: 3},
            {False: 1, True: 4},
        ],
    },
    {
        "model__kernel": ["rbf"],
        "model__C": [0.1, 0.5, 1.0, 2.0, 5.0, 10.0],
        "model__gamma": ["scale", "auto", 0.001, 0.01, 0.05, 0.1],
        "model__class_weight": [
            None,
            "balanced",
            {False: 1, True: 2},
            {False: 1, True: 3},
            {False: 1, True: 4},
        ],
    },
]

# --- 4. Configure and Fit GridSearchCV Exclusively on Training Data ---
# Fit strictly on X_train to prevent validation data leakage; refit=True automatically
# retrains the single winning kernel configuration on the full training set
svm_search = GridSearchCV(
    estimator=svm_pipe,
    param_grid=param_grid,
    scoring=f1_purchase,
    cv=cv,
    refit=True,
    return_train_score=True,
    n_jobs=-1,
)
svm_search.fit(X_train, y_train)

# --- 5. Cross-Validation Kernel Comparison & Winning Summary ---
# Compare the top CV validation F1 achieved by each kernel before inspecting the winner
cv_df = pd.DataFrame(svm_search.cv_results_)
best_per_kernel = cv_df.groupby("param_model__kernel")["mean_test_score"].max()
print(f"Best Linear Kernel CV F1: {best_per_kernel['linear']:.3f}")
print(f"Best RBF Kernel CV F1:    {best_per_kernel['rbf']:.3f}\n")

best_index = svm_search.best_index_
best_cv_train_f1 = svm_search.cv_results_["mean_train_score"][best_index]
best_cv_valid_f1 = svm_search.best_score_

print("Best Overall SVM Parameters:", svm_search.best_params_)
print(f"Best Mean CV Training F1:    {best_cv_train_f1:.3f}")
print(f"Best Mean CV Validation F1:  {best_cv_valid_f1:.3f}")

# --- 6. Support Vector Count Inspection ---
# Extract the single refitted winning SVM pipeline and display support vectors per class
best_svm_pipe = svm_search.best_estimator_
svm_model = best_svm_pipe.named_steps["model"]
print(f"Winning ({svm_model.kernel}) SVM support vectors:", svm_model.n_support_)

Best Linear Kernel CV F1: 0.682
Best RBF Kernel CV F1:    0.699

Best Overall SVM Parameters: {'model__C': 2.0, 'model__class_weight': {False: 1, True: 2}, 'model__gamma': 'scale', 'model__kernel': 'rbf'}
Best Mean CV Training F1:    0.775
Best Mean CV Validation F1:  0.699
Winning (rbf) SVM support vectors: [1404  697]


In [24]:
# --- 1. Generate Predictions on Training and Holdout Sets ---
# Use the single refitted best SVM pipeline for class predictions
y_train_pred = best_svm_pipe.predict(X_train)
y_valid_pred = best_svm_pipe.predict(X_valid)

# --- 2. Build Classification Metrics Comparison Table ---
# Evaluate Accuracy, Precision, Recall, and F1 Score side-by-side on train vs. validation
metrics_comparison = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score"],
    "Training": [
        accuracy_score(y_train, y_train_pred),
        precision_score(y_train, y_train_pred, zero_division=0),
        recall_score(y_train, y_train_pred, zero_division=0),
        f1_score(y_train, y_train_pred, zero_division=0),
    ],
    "Validation": [
        accuracy_score(y_valid, y_valid_pred),
        precision_score(y_valid, y_valid_pred, zero_division=0),
        recall_score(y_valid, y_valid_pred, zero_division=0),
        f1_score(y_valid, y_valid_pred, zero_division=0),
    ],
})

display(metrics_comparison.round(3))

,Metric,Training,Validation
0,Accuracy,0.925,0.887
1,Precision,0.726,0.615
2,Recall,0.831,0.720
3,F1 Score,0.775,0.663


## Decision Tree

In [26]:
# --- 1. Cross-Validation & Scoring Setup ---
# Maintain class proportions (~84.5% False / ~15.5% True) across 5 folds
# and optimize specifically for the positive class (Revenue=True) F1 score
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
f1_purchase = make_scorer(f1_score, pos_label=True, zero_division=0)

# --- 2. Dedicated Decision Tree Base Pipeline ---
# Clone the global preprocessor so fitting GridSearchCV never mutates shared pipelines
dt_pipe = Pipeline([
    ("prep", clone(preprocessor)),
    ("model", DecisionTreeClassifier(random_state=42)),
])

# --- 3. Impurity, Pre/Post-Pruning, & Class-Weight Hyperparameter Grid ---
# Sweep splitting criteria, tree depth, minimum leaf samples, and cost-complexity pruning
# alongside custom class_weight ratios to balance precision and recall under class imbalance
param_grid = {
    "model__criterion": ["gini", "entropy"],
    "model__max_depth": [3, 4, 5, 6, 8, 10, None],
    "model__min_samples_leaf": [5, 10, 15, 25, 50],
    "model__ccp_alpha": [0.0, 0.0005, 0.001, 0.005],
    "model__class_weight": [
        None,
        "balanced",
        {False: 1, True: 2},
        {False: 1, True: 3},
        {False: 1, True: 4},
    ],
}

# --- 4. Configure and Fit GridSearchCV Exclusively on Training Data ---
# Fit strictly on X_train to prevent validation data leakage; refit=True automatically
# retrains the winning hyperparameter combination on the full training set
dt_search = GridSearchCV(
    estimator=dt_pipe,
    param_grid=param_grid,
    scoring=f1_purchase,
    cv=cv,
    refit=True,
    return_train_score=True,
    n_jobs=-1,
)
dt_search.fit(X_train, y_train)

# --- 5. Cross-Validation Performance Summary ---
# Compare mean CV training vs. validation F1 to check generalization before holdout evaluation
best_index = dt_search.best_index_
best_cv_train_f1 = dt_search.cv_results_["mean_train_score"][best_index]
best_cv_valid_f1 = dt_search.best_score_

print("Best Decision Tree Parameters:", dt_search.best_params_)
print(f"Best Mean CV Training F1:     {best_cv_train_f1:.3f}")
print(f"Best Mean CV Validation F1:   {best_cv_valid_f1:.3f}")

# --- 6. Learned Feature Importance Inspection ---
# Extract the refitted winning pipeline and display the top 10 most important features
best_dt_pipe = dt_search.best_estimator_
dt_model = best_dt_pipe.named_steps["model"]

dt_importance = pd.DataFrame({
    "Feature": best_dt_pipe.named_steps["prep"].get_feature_names_out(),
    "Importance": dt_model.feature_importances_,
})

display(
    dt_importance.sort_values(
        "Importance",
        ascending=False,
    ).head(10).round(3)
)

Best Decision Tree Parameters: {'model__ccp_alpha': 0.001, 'model__class_weight': {False: 1, True: 2}, 'model__criterion': 'entropy', 'model__max_depth': 6, 'model__min_samples_leaf': 50}
Best Mean CV Training F1:     0.716
Best Mean CV Validation F1:   0.695


,Feature,Importance
8,num__PageValues,0.766
17,cat__Month_Nov,0.066
6,num__BounceRates,0.033
5,num__ProductRelated_Duration,0.027
16,cat__Month_May,0.026
0,num__Administrative,0.021
15,cat__Month_Mar,0.021
7,num__ExitRates,0.015
1,num__Administrative_Duration,0.015
3,num__Informational_Duration,0.003


In [27]:
# --- 1. Generate Predictions on Training and Holdout Sets ---
# Use the refitted best Decision Tree pipeline for class and probability predictions
y_train_pred = best_dt_pipe.predict(X_train)
y_valid_pred = best_dt_pipe.predict(X_valid)
y_valid_prob = best_dt_pipe.predict_proba(X_valid)[:, 1]

# --- 2. Build Classification Metrics Comparison Table ---
# Evaluate Accuracy, Precision, Recall, and F1 Score side-by-side on train vs. validation
metrics_comparison = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score"],
    "Training": [
        accuracy_score(y_train, y_train_pred),
        precision_score(y_train, y_train_pred, zero_division=0),
        recall_score(y_train, y_train_pred, zero_division=0),
        f1_score(y_train, y_train_pred, zero_division=0),
    ],
    "Validation": [
        accuracy_score(y_valid, y_valid_pred),
        precision_score(y_valid, y_valid_pred, zero_division=0),
        recall_score(y_valid, y_valid_pred, zero_division=0),
        f1_score(y_valid, y_valid_pred, zero_division=0),
    ],
})

display(metrics_comparison.round(3))

,Metric,Training,Validation
0,Accuracy,0.905,0.881
1,Precision,0.669,0.599
2,Recall,0.760,0.696
3,F1 Score,0.711,0.644


## Random Forest

In [28]:
# --- 1. Cross-Validation & Scoring Setup ---
# Maintain class proportions (~84.5% False / ~15.5% True) across 5 folds
# and optimize specifically for the positive class (Revenue=True) F1 score
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
f1_purchase = make_scorer(f1_score, pos_label=True, zero_division=0)

# --- 2. Dedicated Random Forest Base Pipeline ---
# Clone the global preprocessor so fitting GridSearchCV never mutates shared pipelines
rf_pipe = Pipeline([
    ("prep", clone(preprocessor)),
    ("model", RandomForestClassifier(random_state=42, n_jobs=-1)),
])

# --- 3. Regularization, Feature Subsampling, & Class-Weight Hyperparameter Grid ---
# Sweep n_estimators, max_depth, and min_samples_leaf to prevent unpruned overfitting.
# Tune max_features across mixed continuous/one-hot columns and sweep class_weight
# options to balance precision and recall on the imbalanced target.
param_grid = {
    "model__n_estimators": [200, 350],
    "model__max_depth": [6, 10, 14, None],
    "model__min_samples_leaf": [2, 5, 10, 20],
    "model__max_features": ["sqrt", 0.3, 0.5],
    "model__class_weight": [
        None,
        "balanced",
        "balanced_subsample",
        {False: 1, True: 2},
        {False: 1, True: 3},
    ],
}

# --- 4. Configure and Fit GridSearchCV Exclusively on Training Data ---
# Fit strictly on X_train to prevent validation data leakage; refit=True automatically
# retrains the winning hyperparameter combination on the full training set
rf_search = GridSearchCV(
    estimator=rf_pipe,
    param_grid=param_grid,
    scoring=f1_purchase,
    cv=cv,
    refit=True,
    return_train_score=True,
    n_jobs=-1,
)
rf_search.fit(X_train, y_train)

# --- 5. Cross-Validation Performance Summary ---
# Compare mean CV training vs. validation F1 to check generalization before holdout evaluation
best_index = rf_search.best_index_
best_cv_train_f1 = rf_search.cv_results_["mean_train_score"][best_index]
best_cv_valid_f1 = rf_search.best_score_

print("Best Random Forest Parameters:", rf_search.best_params_)
print(f"Best Mean CV Training F1:     {best_cv_train_f1:.3f}")
print(f"Best Mean CV Validation F1:   {best_cv_valid_f1:.3f}")

# --- 6. Learned Feature Importance Inspection ---
# Extract the refitted winning pipeline and display the top 10 most important features
best_rf_pipe = rf_search.best_estimator_
feature_names = best_rf_pipe.named_steps["prep"].get_feature_names_out()
importances = best_rf_pipe.named_steps["model"].feature_importances_

rf_importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances,
})

display(
    rf_importance.sort_values(
        "Importance",
        ascending=False,
    ).head(10).round(3)
)

Best Random Forest Parameters: {'model__class_weight': {False: 1, True: 2}, 'model__max_depth': 14, 'model__max_features': 0.5, 'model__min_samples_leaf': 10, 'model__n_estimators': 350}
Best Mean CV Training F1:     0.784
Best Mean CV Validation F1:   0.711


,Feature,Importance
8,num__PageValues,0.655
7,num__ExitRates,0.077
5,num__ProductRelated_Duration,0.053
4,num__ProductRelated,0.040
6,num__BounceRates,0.031
17,cat__Month_Nov,0.030
1,num__Administrative_Duration,0.026
0,num__Administrative,0.023
3,num__Informational_Duration,0.008
16,cat__Month_May,0.006


In [29]:
# --- 1. Generate Predictions on Training and Holdout Sets ---
# Use the refitted best Random Forest pipeline for class and probability predictions
y_train_pred = best_rf_pipe.predict(X_train)
y_valid_pred = best_rf_pipe.predict(X_valid)
y_valid_prob = best_rf_pipe.predict_proba(X_valid)[:, 1]

# --- 2. Build Classification Metrics Comparison Table ---
# Evaluate Accuracy, Precision, Recall, and F1 Score side-by-side on train vs. validation
metrics_comparison = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score"],
    "Training": [
        accuracy_score(y_train, y_train_pred),
        precision_score(y_train, y_train_pred, zero_division=0),
        recall_score(y_train, y_train_pred, zero_division=0),
        f1_score(y_train, y_train_pred, zero_division=0),
    ],
    "Validation": [
        accuracy_score(y_valid, y_valid_pred),
        precision_score(y_valid, y_valid_pred, zero_division=0),
        recall_score(y_valid, y_valid_pred, zero_division=0),
        f1_score(y_valid, y_valid_pred, zero_division=0),
    ],
})

display(metrics_comparison.round(3))

,Metric,Training,Validation
0,Accuracy,0.930,0.888
1,Precision,0.749,0.621
2,Recall,0.827,0.710
3,F1 Score,0.786,0.662


## Gradient Boost

In [30]:
# --- 1. Cross-Validation & Scoring Setup ---
# Maintain class proportions (~84.5% False / ~15.5% True) across 5 folds
# and optimize specifically for the positive class (Revenue=True) F1 score
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
f1_purchase = make_scorer(f1_score, pos_label=True, zero_division=0)

# --- 2. Dedicated Gradient Boost Base Pipeline ---
# Clone the global preprocessor so fitting GridSearchCV never mutates shared pipelines
gb_pipe = Pipeline([
    ("prep", clone(preprocessor)),
    ("model", GradientBoostingClassifier(random_state=42)),
])

# --- 3. Boosting Shrinkage, Tree Complexity, & Stochastic Subsampling Grid ---
# Tune learning_rate alongside n_estimators to control sequential step size.
# Sweep max_depth, min_samples_leaf, subsample, and max_features to prevent
# overfitting noisy residuals across the 59 transformed features.
param_grid = {
    "model__n_estimators": [100, 170, 250],
    "model__learning_rate": [0.03, 0.05, 0.1],
    "model__max_depth": [3, 4, 5],
    "model__min_samples_leaf": [5, 15],
    "model__subsample": [0.8, 1.0],
    "model__max_features": ["sqrt", 0.5, None],
}

# --- 4. Configure and Fit GridSearchCV Exclusively on Training Data ---
# Fit strictly on X_train to prevent validation data leakage; refit=True automatically
# retrains the winning hyperparameter combination on the full training set
gb_search = GridSearchCV(
    estimator=gb_pipe,
    param_grid=param_grid,
    scoring=f1_purchase,
    cv=cv,
    refit=True,
    return_train_score=True,
    n_jobs=-1,
)
gb_search.fit(X_train, y_train)

# --- 5. Cross-Validation Performance Summary ---
# Compare mean CV training vs. validation F1 to check generalization before holdout evaluation
best_index = gb_search.best_index_
best_cv_train_f1 = gb_search.cv_results_["mean_train_score"][best_index]
best_cv_valid_f1 = gb_search.best_score_

print("Best Gradient Boost Parameters:", gb_search.best_params_)
print(f"Best Mean CV Training F1:      {best_cv_train_f1:.3f}")
print(f"Best Mean CV Validation F1:    {best_cv_valid_f1:.3f}")

# --- 6. Learned Feature Importance Inspection ---
# Extract the refitted winning pipeline and display the top 10 most important features
best_gb_pipe = gb_search.best_estimator_
gb_model = best_gb_pipe.named_steps["model"]

gb_importance = pd.DataFrame({
    "Feature": best_gb_pipe.named_steps["prep"].get_feature_names_out(),
    "Importance": gb_model.feature_importances_,
})

display(
    gb_importance.sort_values(
        "Importance",
        ascending=False,
    ).head(10).round(3)
)

Best Gradient Boost Parameters: {'model__learning_rate': 0.03, 'model__max_depth': 4, 'model__max_features': 0.5, 'model__min_samples_leaf': 15, 'model__n_estimators': 170, 'model__subsample': 1.0}
Best Mean CV Training F1:      0.754
Best Mean CV Validation F1:    0.685


,Feature,Importance
8,num__PageValues,0.719
7,num__ExitRates,0.053
6,num__BounceRates,0.042
5,num__ProductRelated_Duration,0.035
17,cat__Month_Nov,0.034
0,num__Administrative,0.031
4,num__ProductRelated,0.026
1,num__Administrative_Duration,0.015
55,cat__VisitorType_Returning_Visitor,0.007
16,cat__Month_May,0.005


In [31]:
# --- 1. Generate Predictions on Training and Holdout Sets ---
# Use the refitted best Gradient Boost pipeline for class and probability predictions
y_train_pred = best_gb_pipe.predict(X_train)
y_valid_pred = best_gb_pipe.predict(X_valid)
y_valid_prob = best_gb_pipe.predict_proba(X_valid)[:, 1]

# --- 2. Build Classification Metrics Comparison Table ---
# Evaluate Accuracy, Precision, Recall, and F1 Score side-by-side on train vs. validation
metrics_comparison = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score"],
    "Training": [
        accuracy_score(y_train, y_train_pred),
        precision_score(y_train, y_train_pred, zero_division=0),
        recall_score(y_train, y_train_pred, zero_division=0),
        f1_score(y_train, y_train_pred, zero_division=0),
    ],
    "Validation": [
        accuracy_score(y_valid, y_valid_pred),
        precision_score(y_valid, y_valid_pred, zero_division=0),
        recall_score(y_valid, y_valid_pred, zero_division=0),
        f1_score(y_valid, y_valid_pred, zero_division=0),
    ],
})

display(metrics_comparison.round(3))

,Metric,Training,Validation
0,Accuracy,0.927,0.898
1,Precision,0.804,0.693
2,Recall,0.695,0.615
3,F1 Score,0.746,0.652


## Model Comparison

In [32]:
# --- 1. Register All Tuned Model Pipelines ---
model_results = []

pipelines = {
    "Logistic Regression": best_logistic_pipe,
    "Naïve Bayes": best_bayes_pipe,
    "K-Nearest Neighbors": best_knn_pipe,
    "Support Vector Machine": best_svm_pipe,
    "Decision Tree": best_dt_pipe,
    "Random Forest": best_rf_pipe,
    "Gradient Boost": best_gb_pipe,
}

# --- 2. Configure Cross-Validation & Evaluation Metric ---
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
f1_purchase = make_scorer(f1_score, pos_label=True, zero_division=0)

# --- 3. Evaluate All Tuned Models Across Identical Folds ---
for model, pipeline in pipelines.items():
    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring=f1_purchase,
        return_train_score=True,
    )

    model_results.append({
        "Model": model,
        "Mean training F1": scores["train_score"].mean(),
        "Mean CV F1": scores["test_score"].mean(),
    })

display(pd.DataFrame(model_results).round(3))

,Model,Mean training F1,Mean CV F1
0,Logistic Regression,0.694,0.692
1,Naïve Bayes,0.648,0.651
2,K-Nearest Neighbors,1.000,0.665
3,Support Vector Machine,0.775,0.699
4,Decision Tree,0.716,0.695
5,Random Forest,0.784,0.711
6,Gradient Boost,0.754,0.685
